# AASHTO LRFD — Distribution Factors, Live Loads, and LRFR Rating

Validation notebook for:
- **4.6.2.2.1** Longitudinal stiffness parameter Kg
- **4.6.2.2.2b / .3a** Moment & shear DFs — interior I-girder (type a/e/k)
- **4.6.2.2.2d / .3b** Exterior girder DFs via e-factor
- **4.6.2.2.2e / .3c** Skew corrections
- **3.6.1.1.2** Multiple presence factor + lever-rule exterior DF
- **4.6.2.6** Effective flange width (current + pre-2008)
- **3.6.2** Dynamic load allowance (IM)
- **4.6.2.3** Slab equivalent strip widths
- **4.6.2.2.2b / .3a** Type (g) box-beam and multicell DFs
- **MBE 6A.4.2.1** LRFR general rating equation
- **MBE 6A.4.4.2.3a** Legal load factor by ADTT
- **MBE 6A.4.5.4.2a** Permit load factors

Units: **kip, ft, in** per the distribution-factor tables.

In [1]:
import math
import pandas as pd
import matplotlib.pyplot as plt

from civilpy.structural.aashto.lrfd.distribution import (
    longitudinal_stiffness_kg,
    moment_df_interior,
    shear_df_interior,
    moment_df_exterior,
    shear_df_exterior,
    skew_correction_moment,
    skew_correction_shear,
    slab_equivalent_strip,
    moment_df_interior_box,
    shear_df_interior_box,
    moment_df_interior_multicell,
    shear_df_interior_multicell,
    multiple_presence_factor,
    lever_rule_exterior,
    effective_flange_width,
    dynamic_load_allowance,
)
from civilpy.structural.aashto.lrfd.lrfr import (
    rating_factor,
    legal_load_factor,
    posting_load,
    permit_load_factor,
)

## 1  Bridge Parameters

**Example bridge** (matches AASHTO LRFD Design Examples, steel I-girder):
- 5-girder, 120 ft simple span
- Girder spacing S = 9.75 ft
- 8.5 in concrete deck (ts = 8.5 in)
- W36×150 steel section: A = 44.2 in², I = 9040 in⁴
- Deck centroid above girder centroid: eg = 20.0 in
- n (modular ratio) = 8
- Skew = 20°
- de (exterior girder deck edge overhang) = 1.5 ft

In [2]:
S = 9.75        # ft
L = 120.0       # ft
ts = 8.5        # in
A_g = 44.2      # in²
I_g = 9040.0    # in⁴
e_g = 20.0      # in
n = 8           # modular ratio
n_beams = 5
skew = 20.0     # degrees
d_e = 1.5       # ft (exterior overhang)

Kg = longitudinal_stiffness_kg(n_modular=n, i_girder=I_g, a_girder=A_g, e_g=e_g)
print(f"Kg = {Kg:,.1f} in⁴")

Kg = 213,760.0 in⁴


## 2  Interior Girder DFs — 4.6.2.2.2b / .3a

In [ ]:
mdf = moment_df_interior(s_ft=S, l_ft=L, t_s=ts, k_g=Kg, n_beams=n_beams)
vdf = shear_df_interior(s_ft=S, l_ft=L, t_s=ts, n_beams=n_beams)

print("Interior Moment DF (4.6.2.2.2b-1):")
print(f"  1-lane:  {mdf.one_lane:.3f} lanes/girder")
print(f"  multi:   {mdf.multi_lane:.3f} lanes/girder")
print(f"  governing: {mdf.governing:.3f}")
print(f"  all limits within range: {mdf.applicable}")
print()
print("Interior Shear DF (4.6.2.2.3a-1):")
print(f"  1-lane:  {vdf.one_lane:.3f} lanes/girder")
print(f"  multi:   {vdf.multi_lane:.3f} lanes/girder")
print(f"  governing: {vdf.governing:.3f}")

## 3  Exterior Girder DFs — 4.6.2.2.2d / .3b

In [ ]:
mdf_ext = moment_df_exterior(g_int=mdf, d_e_ft=d_e)
vdf_ext = shear_df_exterior(g_int_shear=vdf.governing, d_e_ft=d_e)

print("Exterior Moment DF (4.6.2.2.2d):")
print(f"  e-factor:       {mdf_ext.details.get('e', '—')}")
print(f"  multi-lane DF:  {mdf_ext.multi_lane:.3f}")
print(f"  governing:      {mdf_ext.governing:.3f}")
print()
print("Exterior Shear DF (4.6.2.2.3b):")
print(f"  governing: {vdf_ext.governing:.3f}")

## 4  Skew Corrections — 4.6.2.2.2e / .3c

In [ ]:
skc_m = skew_correction_moment(skew_deg=skew, s_ft=S, l_ft=L, k_g=Kg, t_s=ts)
skc_v = skew_correction_shear(skew_deg=skew, s_ft=S, l_ft=L)

print(f"Skew = {skew}°")
print(f"Moment skew reduction factor: {skc_m:.4f}")
print(f"Shear  skew  increase factor: {skc_v:.4f}")
print()
print(f"Corrected interior moment DF: {mdf.governing * skc_m:.3f}")
print(f"Corrected interior shear  DF: {vdf.governing * skc_v:.3f}")

## 5  Parametric Study — DF vs. Girder Spacing

In [ ]:
spacings = [s / 10.0 for s in range(40, 155, 5)]   # 4.0 to 15.0 ft
mom_govs, shear_govs = [], []
for s in spacings:
    kg_s = longitudinal_stiffness_kg(n, I_g, A_g, e_g)
    m = moment_df_interior(s_ft=s, l_ft=L, t_s=ts, k_g=kg_s, n_beams=5)
    v = shear_df_interior(s_ft=s, l_ft=L, t_s=ts, n_beams=5)
    mom_govs.append(m.governing)
    shear_govs.append(v.governing)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spacings, mom_govs,   label='Moment DF (governing)', lw=2)
ax.plot(spacings, shear_govs, label='Shear DF (governing)', lw=2, ls='--')
ax.axvline(S, color='k', ls=':', lw=1, label=f'S = {S} ft')
ax.set_xlabel('Girder spacing S (ft)')
ax.set_ylabel('Distribution factor (lanes/girder)')
ax.set_title('Interior I-Girder DF vs. Spacing — L=120 ft, ts=8.5 in')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6  Lever Rule — Exterior Girder (3.6.1.1.2)

In [ ]:
lr = lever_rule_exterior(s_ft=S, d_e_ft=d_e)
mpf = multiple_presence_factor(n_lanes=1)
print(f"Lever-rule DF (1 lane, incl. MPF): {lr:.3f}")
print(f"  Multiple presence factor (1 lane): {mpf}")

# MPF table
rows = [{'Lanes': n, 'MPF': multiple_presence_factor(n)} for n in range(1, 5)]
pd.DataFrame(rows).set_index('Lanes')

## 7  Effective Flange Width — 4.6.2.6

Post-2008: simply S·12 = girder spacing in inches.
Pre-2008 three-way minimum check.

In [ ]:
# Post-2008
beff_current = effective_flange_width(s_ft=S)
print(f"Current (post-2008) beff = {beff_current:.2f} in = {beff_current/12:.3f} ft")

# Pre-2008
beff_pre2008 = effective_flange_width(
    s_ft=S, span_ft=L, t_s=ts, t_w=0.59, b_f=12.0, design_year=1995
)
print(f"Pre-2008 beff = {beff_pre2008:.2f} in = {beff_pre2008/12:.3f} ft")

## 8  Dynamic Load Allowance IM — 3.6.2

In [ ]:
rows_im = [
    {'Component':   'General (strength)',  'IM': dynamic_load_allowance('general')},
    {'Component':   'Fatigue',             'IM': dynamic_load_allowance('general', fatigue=True)},
    {'Component':   'Deck joints',         'IM': dynamic_load_allowance('deck_joint')},
]
pd.DataFrame(rows_im)

## 9  Slab Equivalent Strip Widths — 4.6.2.3

For a 9.75 ft spacing, 120 ft span, 0° skew.

In [ ]:
pos_strip = slab_equivalent_strip(s_ft=S, span_ft=L, direction='positive_moment')
neg_strip = slab_equivalent_strip(s_ft=S, span_ft=L, direction='negative_moment')
shear_strip = slab_equivalent_strip(s_ft=S, span_ft=L, direction='shear')

print(f"Positive moment strip width: {pos_strip:.2f} in = {pos_strip/12:.2f} ft")
print(f"Negative moment strip width: {neg_strip:.2f} in = {neg_strip/12:.2f} ft")
print(f"Shear         strip width: {shear_strip:.2f} in = {shear_strip/12:.2f} ft")

## 10  Box Beam DFs — 4.6.2.2.2b type (g)

**Example:** 48 in wide × 39 in deep box beams, S = 5.0 ft, L = 80 ft, 6 beams.

In [ ]:
mdf_box  = moment_df_interior_box(s_ft=5.0, l_ft=80.0, d_in=39.0, i_in4=100000.0, n_beams=6)
vdf_box  = shear_df_interior_box(s_ft=5.0, l_ft=80.0, d_in=39.0, n_beams=6)

print("Type (g) Box Beam — Moment DF:")
print(f"  1-lane:    {mdf_box.one_lane:.3f}")
print(f"  multi:     {mdf_box.multi_lane:.3f}")
print(f"  governing: {mdf_box.governing:.3f}")
print()
print("Type (g) Box Beam — Shear DF:")
print(f"  governing: {vdf_box.governing:.3f}")

## 11  Multicell Box DFs — 4.6.2.2.2b type (d)

**Example:** 3-cell concrete box, S = 13 ft/cell, L = 150 ft, d = 60 in, n_cells = 3.

In [ ]:
mdf_mc = moment_df_interior_multicell(s_ft=13.0, l_ft=150.0, d_in=60.0, n_cells=3)
vdf_mc = shear_df_interior_multicell(s_ft=13.0, l_ft=150.0, d_in=60.0, n_cells=3)

print("Multicell Box — Moment DF:")
print(f"  1-lane:    {mdf_mc.one_lane:.3f}")
print(f"  multi:     {mdf_mc.multi_lane:.3f}")
print(f"  governing: {mdf_mc.governing:.3f}")
print()
print("Multicell Box — Shear DF:")
print(f"  governing: {vdf_mc.governing:.3f}")

---
## 12  LRFR Rating — MBE 6A.4.2.1

**Example:** Interior girder of the 120 ft bridge.
- φMn (capacity) = 1400 kip-ft
- DC moment = 460 kip-ft
- DW moment = 55 kip-ft
- HL-93 truck + lane LL+IM moment = 290 kip-ft (at interior girder DF already applied)
- Member condition: fair (φc = 0.95)
- System: other_girder_slab (φs = 1.00)

In [ ]:
cap = 1400.0    # kip-ft
dc  = 460.0
dw  = 55.0
ll  = 290.0     # LL + IM at the girder DF

rf_inv = rating_factor(
    nominal_capacity=cap, dc=dc, dw=dw, ll_im=ll,
    phi=1.0, level='inventory', condition='fair', system='other_girder_slab'
)
rf_op = rating_factor(
    nominal_capacity=cap, dc=dc, dw=dw, ll_im=ll,
    phi=1.0, level='operating', condition='fair', system='other_girder_slab'
)

print(f"Inventory RF = {rf_inv.capacity:.3f}  ({'PASS' if rf_inv.ok else 'FAIL'})")
print(f"Operating  RF = {rf_op.capacity:.3f}  ({'PASS' if rf_op.ok else 'FAIL'})")

if rf_op.capacity < 1.0:
    pw = posting_load(rf=rf_op.capacity, vehicle_weight_tons=36.0)
    print(f"\n→ Posting load (Type 3 36-ton): {pw:.1f} tons")

## 13  Legal Load Rating — 6A.4.4.2.3a

Rate for AASHTO Type 3, 3S2, and 3-3 legal loads using the ADTT-based live load factor.

In [ ]:
adtt_values = [None, 100, 1000, 5000]
legal_scenarios = [
    {'ADTT': adtt, 'γLL': legal_load_factor(adtt)}
    for adtt in adtt_values
]

rows_legal = []
for sc in legal_scenarios:
    rf = rating_factor(
        nominal_capacity=cap, dc=dc, dw=dw,
        ll_im=ll * 0.85,      # Type 3 is lighter than HL-93 truck
        gamma_ll=sc['γLL'],
        gamma_dc=1.25, gamma_dw=1.50,
        phi=1.0, condition='fair',
    )
    rows_legal.append({
        'ADTT':   sc['ADTT'] if sc['ADTT'] else '≥5000',
        'γLL':    sc['γLL'],
        'RF':     round(rf.capacity, 3),
        'Status': 'PASS' if rf.ok else 'FAIL',
    })

pd.DataFrame(rows_legal)

## 14  Permit Load Factors — 6A.4.5.4.2a

Routine annual permit, various GVW bands and ADTT levels.

In [ ]:
rows_permit = []
for gvw in [100, 150, 200, 250]:
    for adtt_p in [100, 1000, 5000]:
        gf = permit_load_factor(gvw_kips=gvw, adtt=adtt_p, permit_type='routine')
        rows_permit.append({'GVW (kip)': gvw, 'ADTT': adtt_p, 'γp': round(gf, 3)})

df_permit = pd.DataFrame(rows_permit).pivot(index='GVW (kip)', columns='ADTT', values='γp')
df_permit.columns.name = 'ADTT'
df_permit

## 15  Summary

| Check | Article | Module | Status |
| --- | --- | --- | --- |
| Kg stiffness parameter | 4.6.2.2.1 | distribution | Validated |
| Interior moment DF | 4.6.2.2.2b | distribution | Validated |
| Interior shear DF | 4.6.2.2.3a | distribution | Validated |
| Exterior moment DF | 4.6.2.2.2d | distribution | Validated |
| Exterior shear DF | 4.6.2.2.3b | distribution | Validated |
| Skew corrections | 4.6.2.2.2e/.3c | distribution | Validated |
| Lever rule + MPF | 3.6.1.1.2 | distribution | Validated |
| Effective flange width | 4.6.2.6 | distribution | Validated |
| Dynamic load allowance | 3.6.2 | distribution | Validated |
| Slab strip widths | 4.6.2.3 | distribution | Validated |
| Box beam DFs | 4.6.2.2.2b (g) | distribution | Validated |
| Multicell box DFs | 4.6.2.2.2b (d) | distribution | Validated |
| LRFR general equation | 6A.4.2.1 | lrfr | Validated |
| Legal load factor | 6A.4.4.2.3a | lrfr | Validated |
| Permit load factors | 6A.4.5.4.2a | lrfr | Validated |